# VIMS background spectra — ansa comparison

Loads all `17_VIMS_campain/*.json` files (Background pixels collected in notebook 17),
groups them by ansa, and produces:

1. **Master spectra** — median I/F per wavelength
2. **Test 1** — continuum I/F at 2.2 µm vs ansa (Hedman column-density test)
3. **Test 2** — 3.0 µm water-ice band depth vs ansa (microphysics test)
4. **Thesis figure** — three-panel composite for Section 6.3

> **Note on the 1.5 µm / 2.0 µm ice bands**: these absorption features are characteristic of
> coarse-grained water ice and arise from bulk absorption over many mean-free paths.
> For the E ring's sub-micron to few-micron particles the spectrum is dominated by single
> scattering, and the 1.5 / 2.0 µm features are not detectable — consistent with the
> literature (Hedman et al. 2009, Cuzzi et al. 2018).  The 3.0 µm feature
> (OH-stretch Fresnel reflection) is the only robust water-ice signature in this regime.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import defaultdict

In [ ]:
CAMPAIGN_DIR = Path("/home/arnaudm/projects/LuminousBandsInERing/mnt-storage/post_pipeline/17_VIMS_campain")
OUT_DIR      = Path("/home/arnaudm/projects/LuminousBandsInERing/mnt-storage/post_pipeline/18_ansa_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANSAE  = ["subsolar", "antisolar", "morning", "evening"]
COLORS = {"subsolar": "#e6a817", "antisolar": "#1a3a6b",
          "morning":  "#1b7a5e", "evening":   "#b5341a"}
LABELS = {"subsolar": "Sub-solar", "antisolar": "Anti-solar",
          "morning":  "Morning",   "evening":   "Evening"}

# Test 1: continuum wavelength
WL_CONTINUUM = 2.20   # µm

# Test 2: 3.0 µm water-ice band
# Continuum shortward of the 2.7-3.1 µm absorption complex
# Band minimum in the deepest part of the feature
WL_30_CTN = 2.60   # µm  (clean continuum before the feature)
WL_30_MIN = 2.88   # µm  (band minimum)

## Load and group JSON files

In [ ]:
# ansa → list of (wvlns_array, spec_array) for every background pixel
raw: dict[str, list[tuple[np.ndarray, np.ndarray]]] = defaultdict(list)

for jpath in sorted(CAMPAIGN_DIR.glob("*.json")):
    data = json.loads(jpath.read_text())
    ansa = data["ansa"]
    for entry in data["regions"]["Background"]:
        wvlns = np.array(entry["wvlns"], dtype=np.float64)
        spec  = np.array([v if v is not None else np.nan
                          for v in entry["spec"]], dtype=np.float64)
        raw[ansa].append((wvlns, spec))

print("Background spectra loaded per ansa:")
for a in ANSAE:
    print(f"  {LABELS[a]:12s}: {len(raw[a]):4d} pixels")

## Compute master spectra

In [ ]:
def nearest_idx(wvlns: np.ndarray, target: float) -> int:
    return int(np.argmin(np.abs(wvlns - target)))


masters: dict[str, dict] = {}

for ansa in ANSAE:
    entries = raw[ansa]
    if not entries:
        print(f"  {ansa}: no data")
        continue

    wvlns_ref = entries[0][0]   # reference grid (all cubes share the same 256-channel IR grid)
    spectra   = np.array([s for w, s in entries if len(w) == len(wvlns_ref)])

    med = np.nanmedian(spectra, axis=0)
    p16 = np.nanpercentile(spectra, 16, axis=0)
    p84 = np.nanpercentile(spectra, 84, axis=0)

    masters[ansa] = {"wvlns": wvlns_ref, "med": med, "p16": p16, "p84": p84,
                     "n": spectra.shape[0]}

    print(f"  {LABELS[ansa]:12s}: {spectra.shape[0]:4d} spectra  "
          f"({spectra.shape[1]} channels, "
          f"{wvlns_ref[0]:.3f}–{wvlns_ref[-1]:.3f} µm)")

## Test 1 — Continuum I/F at 2.2 µm vs ansa

Tests the Hedman et al. (2012) prediction: the Morning ansa should show lower
I/F due to reduced particle column density in Saturn's shadow.

In [ ]:
continuum_med = {}
continuum_err = {}   # half-width (p84 - p16) / 2 at that channel

for ansa, m in masters.items():
    i = nearest_idx(m["wvlns"], WL_CONTINUUM)
    continuum_med[ansa] = float(m["med"][i])
    continuum_err[ansa] = float((m["p84"][i] - m["p16"][i]) / 2)

ref_wvlns = masters[ANSAE[0]]["wvlns"]
i_ctn = nearest_idx(ref_wvlns, WL_CONTINUUM)
print(f"Using channel {i_ctn}: λ = {ref_wvlns[i_ctn]:.4f} µm  (target {WL_CONTINUUM} µm)")
print()
print(f"Continuum I/F at {ref_wvlns[i_ctn]:.4f} µm:")
for a in ANSAE:
    if a in continuum_med:
        print(f"  {LABELS[a]:12s}: {continuum_med[a]:.5f}  ±{continuum_err[a]:.5f}")

## Test 2 — 3.0 µm water-ice band depth vs ansa

The 1.5 µm and 2.0 µm absorption features require many mean-free paths through bulk ice
and are absent for sub-micron E ring grains.  The 3.0 µm feature
(OH-stretch Fresnel reflection + absorption) is detectable even for very fine particles.

$$D_{3.0} = 1 - \frac{(I/F)_{2.88\,\mu\text{m}}}{(I/F)_{2.60\,\mu\text{m}}}$$

Deeper band → stronger ice signature → consistent with smaller or more crystalline grains.

In [ ]:
def band_depth_array(spec_arr: np.ndarray, wvlns: np.ndarray,
                     wl_ctn: float, wl_min: float) -> np.ndarray:
    """Per-spectrum band depth D = 1 - I/F_min / I/F_ctn.  Returns NaN where undefined."""
    i_ctn = nearest_idx(wvlns, wl_ctn)
    i_min = nearest_idx(wvlns, wl_min)
    with np.errstate(invalid="ignore", divide="ignore"):
        d = 1.0 - spec_arr[:, i_min] / spec_arr[:, i_ctn]
    d[~np.isfinite(d)] = np.nan
    return d


depth30_med = {}
depth30_err = {}

ref_wvlns = masters[ANSAE[0]]["wvlns"]
i_ctn30 = nearest_idx(ref_wvlns, WL_30_CTN)
i_min30 = nearest_idx(ref_wvlns, WL_30_MIN)
print(f"Continuum channel {i_ctn30}: λ = {ref_wvlns[i_ctn30]:.4f} µm")
print(f"Band min  channel {i_min30}: λ = {ref_wvlns[i_min30]:.4f} µm")
print()

for ansa in ANSAE:
    entries   = raw[ansa]
    if not entries: continue
    wvlns_ref = entries[0][0]
    spectra   = np.array([s for w, s in entries if len(w) == len(wvlns_ref)])
    depths    = band_depth_array(spectra, wvlns_ref, WL_30_CTN, WL_30_MIN)
    depth30_med[ansa] = float(np.nanmedian(depths))
    depth30_err[ansa] = float((np.nanpercentile(depths, 84) - np.nanpercentile(depths, 16)) / 2)

print(f"3.0 µm band depth  (CTN={WL_30_CTN:.2f} µm, MIN={WL_30_MIN:.2f} µm):")
for a in ANSAE:
    if a in depth30_med:
        print(f"  {LABELS[a]:12s}: {depth30_med[a]:.4f}  ±{depth30_err[a]:.4f}")

## Thesis figure — Section 6.3

Three-panel layout:
- **Top** — master spectra with 16–84 pct shaded envelopes (full wavelength range)
- **Bottom-left** — continuum I/F at 2.2 µm (Hedman test)
- **Bottom-right** — 3.0 µm band depth (microphysics test)

In [ ]:
fig, ax_spec = plt.subplots(figsize=(12, 4.5), dpi=150)

for ansa in ANSAE:
    m = masters[ansa]
    ax_spec.plot(m["wvlns"], m["med"], color=COLORS[ansa], lw=1.6,
                 label=f"{LABELS[ansa]}  (n={m['n']})")

ax_spec.set_xlim(0.88, 5.13)
ax_spec.set_ylim(bottom=0)
ax_spec.set_xlabel("Wavelength (µm)", fontsize=10)
ax_spec.set_ylabel("I/F", fontsize=10)
ax_spec.set_title("VIMS background spectra per ansa — median", fontsize=10, pad=6)
ax_spec.legend(fontsize=8, loc="upper right", framealpha=0.85)
ax_spec.grid(True, lw=0.4, alpha=0.45)

ylim_top = ax_spec.get_ylim()[1]
for wl, txt in [(1.25, "1.25"), (1.52, "1.52"), (2.02, "2.02"),
                (2.60, "2.6 CTN"), (2.88, "2.88 MIN"), (3.10, "3.1")]:
    ax_spec.axvline(wl, color="#999999", lw=0.7, ls=":", alpha=0.75)
    ax_spec.text(wl + 0.04, ylim_top * 0.97,
                 txt, fontsize=6, color="#666666", va="top", rotation=90)

fig.tight_layout()
out_path = OUT_DIR / "fig_ansa_spectra.pdf"
fig.savefig(out_path, bbox_inches="tight")
fig.savefig(out_path.with_suffix(".png"), dpi=200, bbox_inches="tight")
print(f"Saved: {out_path.name}  +  .png")
plt.show()


## Zoom: 3.0 µm feature

Same as the top panel but restricted to the 2.4–3.8 µm region where the
water-ice OH-stretch feature is visible.

In [ ]:
fig_z, ax_z = plt.subplots(figsize=(7, 4), dpi=150)

for ansa in ANSAE:
    m    = masters[ansa]
    w    = m["wvlns"]
    mask = (w >= 2.40) & (w <= 3.80)
    ax_z.plot(w[mask], m["med"][mask], color=COLORS[ansa], lw=1.8,
              label=LABELS[ansa])

ax_z.axvline(WL_30_CTN, color="#555", lw=1.0, ls="--", alpha=0.7,
             label=f"CTN {WL_30_CTN:.2f} µm")
ax_z.axvline(WL_30_MIN, color="#111", lw=1.0, ls="--", alpha=0.7,
             label=f"MIN {WL_30_MIN:.2f} µm")
ax_z.set_xlim(2.40, 3.80)
ax_z.set_ylim(bottom=0)
ax_z.set_xlabel("Wavelength (µm)", fontsize=10)
ax_z.set_ylabel("I/F", fontsize=10)
ax_z.set_title("3.0 µm water-ice feature — zoom", fontsize=10)
ax_z.legend(fontsize=8, loc="upper right")
ax_z.grid(True, lw=0.4, alpha=0.45)
fig_z.tight_layout()

out_z = OUT_DIR / "fig_3um_zoom.png"
fig_z.savefig(out_z, dpi=200, bbox_inches="tight")
print(f"Saved: {out_z.name}")
plt.show()


## Numerical summary

## Smoothed version — moving average (window = 8 channels)

In [ ]:
def smooth(arr: np.ndarray, window: int = 8) -> np.ndarray:
    """Centred moving average; edges filled with NaN."""
    out = np.full_like(arr, np.nan, dtype=np.float64)
    half = window // 2
    for i in range(half, len(arr) - half):
        out[i] = np.nanmean(arr[i - half : i + half])
    return out


SMOOTH_WINDOW = 8

fig_s, ax_ss = plt.subplots(figsize=(12, 4.5), dpi=150)

for ansa in ANSAE:
    m = masters[ansa]
    ax_ss.plot(m["wvlns"], smooth(m["med"], SMOOTH_WINDOW), color=COLORS[ansa], lw=1.6,
               label=f"{LABELS[ansa]}  (n={m['n']})")

ax_ss.set_xlim(0.88, 5.13)
ax_ss.set_ylim(bottom=0)
ax_ss.set_xlabel("Wavelength (µm)", fontsize=10)
ax_ss.set_ylabel("I/F", fontsize=10)
ax_ss.set_title(f"VIMS background spectra per ansa — median, moving avg (w={SMOOTH_WINDOW})",
                fontsize=10, pad=6)
ax_ss.legend(fontsize=8, loc="upper right", framealpha=0.85)
ax_ss.grid(True, lw=0.4, alpha=0.45)

ylim_top_s = ax_ss.get_ylim()[1]
for wl, txt in [(1.25, "1.25"), (1.52, "1.52"), (2.02, "2.02"),
                (2.60, "2.6 CTN"), (2.88, "2.88 MIN"), (3.10, "3.1")]:
    ax_ss.axvline(wl, color="#999999", lw=0.7, ls=":", alpha=0.75)
    ax_ss.text(wl + 0.04, ylim_top_s * 0.97,
               txt, fontsize=6, color="#666666", va="top", rotation=90)

fig_s.tight_layout()
out_s = OUT_DIR / "fig_ansa_spectra_smoothed.pdf"
fig_s.savefig(out_s, bbox_inches="tight")
fig_s.savefig(out_s.with_suffix(".png"), dpi=200, bbox_inches="tight")
print(f"Saved: {out_s.name}  +  .png")
plt.show()

# ── Zoom: 3.0 µm feature (smoothed) ─────────────────────────────────────────
fig_zs, ax_zs = plt.subplots(figsize=(7, 4), dpi=150)

for ansa in ANSAE:
    m    = masters[ansa]
    w    = m["wvlns"]
    mask = (w >= 2.40) & (w <= 3.80)
    ax_zs.plot(w[mask], smooth(m["med"], SMOOTH_WINDOW)[mask],
               color=COLORS[ansa], lw=1.8, label=LABELS[ansa])

ax_zs.axvline(WL_30_CTN, color="#555", lw=1.0, ls="--", alpha=0.7,
              label=f"CTN {WL_30_CTN:.2f} µm")
ax_zs.axvline(WL_30_MIN, color="#111", lw=1.0, ls="--", alpha=0.7,
              label=f"MIN {WL_30_MIN:.2f} µm")
ax_zs.set_xlim(2.40, 3.80)
ax_zs.set_ylim(bottom=0)
ax_zs.set_xlabel("Wavelength (µm)", fontsize=10)
ax_zs.set_ylabel("I/F", fontsize=10)
ax_zs.set_title(f"3.0 µm feature — zoom, moving avg (w={SMOOTH_WINDOW})", fontsize=10)
ax_zs.legend(fontsize=8, loc="upper right")
ax_zs.grid(True, lw=0.4, alpha=0.45)
fig_zs.tight_layout()

out_zs = OUT_DIR / "fig_3um_zoom_smoothed.png"
fig_zs.savefig(out_zs, dpi=200, bbox_inches="tight")
print(f"Saved: {out_zs.name}")
plt.show()


In [ ]:
import pandas as pd

rows = []
for a in ANSAE:
    if a not in masters: continue
    rows.append({
        "Ansa":                       LABELS[a],
        "N pixels":                   masters[a]["n"],
        f"I/F @ {WL_CONTINUUM:.2f} µm": f"{continuum_med.get(a, float('nan')):.5f}",
        "± (1σ)": f"{continuum_err.get(a, float('nan')):.5f}",
        "D_3.0":  f"{depth30_med.get(a, float('nan')):.4f}",
        "± (1σ) ": f"{depth30_err.get(a, float('nan')):.4f}",
    })

df = pd.DataFrame(rows).set_index("Ansa")
display(df)

## Supplemental — Exclude noisy flybys (164EN, 139EN, 141EN, 122EN, 123EN)

This section replicates the primary analysis while excluding a set of manually identified noisy flybys: 164EN, 139EN, 141EN, 122EN, and 123EN.

In [ ]:
noisy_flybys = ["164EN", "139EN", "141EN", "122EN", "123EN"]

raw_filtered = defaultdict(list)
for jpath in sorted(CAMPAIGN_DIR.glob("*.json")):
    if any(fb in jpath.name for fb in noisy_flybys):
        continue
    data = json.loads(jpath.read_text())
    ansa = data["ansa"]
    for entry in data["regions"]["Background"]:
        wvlns = np.array(entry["wvlns"], dtype=np.float64)
        spec  = np.array([v if v is not None else np.nan
                          for v in entry["spec"]], dtype=np.float64)
        raw_filtered[ansa].append((wvlns, spec))

print("Background spectra loaded per ansa (filtered):")
for a in ANSAE:
    print(f"  {LABELS[a]:12s}: {len(raw_filtered[a]):4d} pixels")

masters_filtered = {}
for ansa in ANSAE:
    entries = raw_filtered[ansa]
    if not entries:
        continue

    wvlns_ref = entries[0][0]
    spectra   = np.array([s for w, s in entries if len(w) == len(wvlns_ref)])

    masters_filtered[ansa] = {
        "wvlns": wvlns_ref,
        "med": np.nanmedian(spectra, axis=0),
        "p16": np.nanpercentile(spectra, 16, axis=0),
        "p84": np.nanpercentile(spectra, 84, axis=0),
        "n": spectra.shape[0]
    }

# Reproduce the main figure
fig, ax_spec = plt.subplots(figsize=(12, 4.5), dpi=150)
for ansa in ANSAE:
    if ansa not in masters_filtered: continue
    m = masters_filtered[ansa]
    ax_spec.plot(m["wvlns"], m["med"], color=COLORS[ansa], lw=1.6,
                 label=f"{LABELS[ansa]}  (n={m['n']})")

ax_spec.set_xlim(0.88, 5.13)
ax_spec.set_ylim(bottom=0)
ax_spec.set_xlabel("Wavelength (µm)", fontsize=10)
ax_spec.set_ylabel("I/F", fontsize=10)
ax_spec.set_title("VIMS background spectra per ansa — median (filtered)", fontsize=10, pad=6)
ax_spec.legend(fontsize=8, loc="upper right", framealpha=0.85)
ax_spec.grid(True, lw=0.4, alpha=0.45)

out_path = OUT_DIR / "fig_ansa_spectra_filtered.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()

# Zoom 3.0 µm
fig_z, ax_z = plt.subplots(figsize=(7, 4), dpi=150)
for ansa in ANSAE:
    if ansa not in masters_filtered: continue
    m    = masters_filtered[ansa]
    w    = m["wvlns"]
    mask = (w >= 2.40) & (w <= 3.80)
    ax_z.plot(w[mask], m["med"][mask], color=COLORS[ansa], lw=1.8, label=LABELS[ansa])

ax_z.axvline(WL_30_CTN, color="#555", lw=1.0, ls="--", alpha=0.7, label=f"CTN {WL_30_CTN:.2f} µm")
ax_z.axvline(WL_30_MIN, color="#111", lw=1.0, ls="--", alpha=0.7, label=f"MIN {WL_30_MIN:.2f} µm")
ax_z.set_xlim(2.40, 3.80)
ax_z.set_ylim(bottom=0)
ax_z.set_xlabel("Wavelength (µm)", fontsize=10)
ax_z.set_ylabel("I/F", fontsize=10)
ax_z.set_title("3.0 µm water-ice feature — zoom (filtered)", fontsize=10)
ax_z.legend(fontsize=8, loc="upper right")
ax_z.grid(True, lw=0.4, alpha=0.45)
out_z = OUT_DIR / "fig_3um_zoom_filtered.png"
fig_z.savefig(out_z, dpi=200, bbox_inches="tight")
plt.show()

# Smoothed version
fig_s, ax_ss = plt.subplots(figsize=(12, 4.5), dpi=150)
for ansa in ANSAE:
    if ansa not in masters_filtered: continue
    m = masters_filtered[ansa]
    ax_ss.plot(m["wvlns"], smooth(m["med"], SMOOTH_WINDOW), color=COLORS[ansa], lw=1.6,
               label=f"{LABELS[ansa]}  (n={m['n']})")
ax_ss.set_xlim(0.88, 5.13)
ax_ss.set_ylim(bottom=0)
ax_ss.set_xlabel("Wavelength (µm)", fontsize=10)
ax_ss.set_ylabel("I/F", fontsize=10)
ax_ss.set_title(f"VIMS background spectra per ansa — median, moving avg (w={SMOOTH_WINDOW}) (filtered)",
                fontsize=10, pad=6)
ax_ss.legend(fontsize=8, loc="upper right", framealpha=0.85)
ax_ss.grid(True, lw=0.4, alpha=0.45)
plt.show()

# Recompute the Numerical Summary logic for the filtered data
continuum_med_f = {}
continuum_err_f = {}
depth30_med_f = {}
depth30_err_f = {}

for ansa, m in masters_filtered.items():
    i_ctn = nearest_idx(m["wvlns"], WL_CONTINUUM)
    continuum_med_f[ansa] = float(m["med"][i_ctn])
    continuum_err_f[ansa] = float((m["p84"][i_ctn] - m["p16"][i_ctn]) / 2)

for ansa in ANSAE:
    entries = raw_filtered[ansa]
    if not entries: continue
    wvlns_ref = entries[0][0]
    spectra   = np.array([s for w, s in entries if len(w) == len(wvlns_ref)])
    depths    = band_depth_array(spectra, wvlns_ref, WL_30_CTN, WL_30_MIN)
    depth30_med_f[ansa] = float(np.nanmedian(depths))
    depth30_err_f[ansa] = float((np.nanpercentile(depths, 84) - np.nanpercentile(depths, 16)) / 2)

rows_f = []
for a in ANSAE:
    if a not in masters_filtered: continue
    rows_f.append({
        "Ansa":                       LABELS[a],
        "N pixels":                   masters_filtered[a]["n"],
        f"I/F @ {WL_CONTINUUM:.2f} µm": f"{continuum_med_f.get(a, float('nan')):.5f}",
        "± (1σ)": f"{continuum_err_f.get(a, float('nan')):.5f}",
        "D_3.0":  f"{depth30_med_f.get(a, float('nan')):.4f}",
        "± (1σ) ": f"{depth30_err_f.get(a, float('nan')):.4f}",
    })

df_f = pd.DataFrame(rows_f).set_index("Ansa")
display(df_f)